# Pipeline de previsão de volatilidade, TSMixerX

Este notebook clona o repositório `hugogobato/Mestrado_Anna_Julia`, instala as dependências, executa o pipeline e empacota forecasts, pesos dos modelos, configurações e métricas para download. O modo curto valida o fluxo; o modo full deve ser usado no runtime GPU do Colab.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/hugogobato/Mestrado_Anna_Julia.git'
ROOT = Path('/content/Mestrado_Anna_Julia')
if not (ROOT / 'src').exists():
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Projeto:', ROOT)
print('CUDA disponível:', end=' ')
try:
    import torch
    print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
except Exception as exc:
    print('não verificado:', exc)

## Dados

A base Excel é versionada junto com o projeto. Se o repositório estiver sem a base, esta célula permite fazer upload manual no Colab.

In [ ]:
DATA_FILE = ROOT / 'base de dados dissertação.xlsx'
if not DATA_FILE.exists():
    from google.colab import files
    uploaded = files.upload()
    if DATA_FILE.name not in uploaded:
        raise FileNotFoundError(f'Envie o arquivo exatamente como: {DATA_FILE.name}')
print(DATA_FILE, f'{DATA_FILE.stat().st_size / 1e6:.2f} MB')

In [ ]:
RUN_MODE = 'final'  # opções: 'smoke', 'validation', 'final'
if RUN_MODE not in {'smoke', 'validation', 'final'}:
    raise ValueError('RUN_MODE deve ser smoke, validation ou final')
RUN_BENCHMARKS = True
RUN_TSMIXERX = True
RUN_EVALUATION = True
RUN_ROBUSTNESS = RUN_MODE == 'final'
SMOKE = RUN_MODE == 'smoke'
TSMIXERX_TRIALS = {'smoke': 3, 'validation': 20, 'final': 200}[RUN_MODE]
TSMIXERX_MAX_STEPS = {'smoke': 30, 'validation': 200, 'final': 1000}[RUN_MODE]
print(f'RUN_MODE={RUN_MODE} | Optuna trials={TSMIXERX_TRIALS} | max_steps={TSMIXERX_MAX_STEPS}')

def run_script(script, *args):
    command = [sys.executable, str(ROOT / 'src' / script), *map(str, args)]
    print('Executando:', ' '.join(command))
    subprocess.run(command, cwd=ROOT, check=True)

prep_args = ['--smoke'] if RUN_MODE != 'final' else []
run_script('00_data_prep.py', *prep_args)
run_script('01_feature_selection.py')

## Benchmarks e forecasts

Os forecasts são salvos em Parquet e CSV. Os benchmarks treináveis também deixam seus artefatos por janela em `results/models/benchmarks/`.

In [ ]:
if RUN_BENCHMARKS:
    benchmark_args = ['--smoke'] if SMOKE else []
    run_script('02_benchmarks.py', *benchmark_args)

## TSMixerX

Cada janela rolling salva um arquivo `.pt` contendo pesos, escalas, arquitetura, features e datas da janela. A configuração vencedora e o banco Optuna permanecem em `results/hp_search/`.

In [ ]:
if RUN_TSMIXERX:
    args = ['--trials', TSMIXERX_TRIALS, '--max-steps', TSMIXERX_MAX_STEPS]
    if SMOKE:
        args.append('--smoke')
    run_script('03_tsmixerx.py', *args)

In [ ]:
if RUN_EVALUATION:
    run_script('04_evaluate.py', '--mcs-reps', 1000, '--mcs-block-size', 1000, '--mcs-size', 0.05)

if RUN_ROBUSTNESS:
    run_script('05_robustness.py', '--seeds', 42, 123, 2024, '--max-steps', TSMIXERX_MAX_STEPS)

import pandas as pd
for forecast_file in [ROOT / 'results/forecasts_benchmarks.parquet', ROOT / 'results/forecasts_tsmixerx.parquet']:
    if forecast_file.exists():
        forecast = pd.read_parquet(forecast_file)
        print(f'\n{forecast_file.name}: {forecast.shape}')
        display(forecast.head())

metrics_file = ROOT / 'results/metrics.csv'
if metrics_file.exists():
    display(pd.read_csv(metrics_file))

robustness_metrics = ROOT / 'results/robustness/metrics_robustness.csv'
if robustness_metrics.exists():
    display(pd.read_csv(robustness_metrics))

model_files = sorted((ROOT / 'results/models').rglob('*')) if (ROOT / 'results/models').exists() else []
print(f'\nArtefatos de modelos: {len([p for p in model_files if p.is_file()])}')
for path in model_files[:10]:
    if path.is_file():
        print(path.relative_to(ROOT))

In [ ]:
# Exemplo de recarga de um modelo salvo, útil para reutilizar uma janela rolling.
first_artifact = ROOT / 'results/models/tsmixerx/tsmixerx_window_00.pt'
if first_artifact.exists():
    import importlib.util
    spec = importlib.util.spec_from_file_location('tsmixerx_module', ROOT / 'src/03_tsmixerx.py')
    tsmixerx_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(tsmixerx_module)
    loaded_model = tsmixerx_module.TSMixerXRegressor.load_artifact(first_artifact)
    print('Modelo recarregado:', first_artifact.name, '| input_size=', loaded_model.input_size, '| features=', len(loaded_model.x_center))
else:
    print('Nenhum artefato TSMixerX encontrado; execute a célula do modelo primeiro.')

## Exportação de modelos e forecasts

O ZIP inclui `results/` completo, portanto contém forecasts Parquet/CSV, pesos `.pt`/`.joblib`, manifestos, configuração HP, métricas e figuras.

In [ ]:
artifact_manifest = {
    'repo_url': REPO_URL,
    'run_mode': RUN_MODE,
    'smoke': SMOKE,
    'run_robustness': RUN_ROBUSTNESS,
    'tsmixerx_trials': TSMIXERX_TRIALS,
    'tsmixerx_max_steps': TSMIXERX_MAX_STEPS,
    'hp_validation_scheme': 'fixed chronological 8-year train / 2-year validation; no internal rolling 4+1+1',
    'mcs': {'implementation': 'arch.bootstrap.MCS', 'loss': 'MSE', 'size': 0.05, 'reps': 1000, 'block_size': 1000, 'method': 'R'},
    'files': [str(p.relative_to(ROOT)) for p in (ROOT / 'results').rglob('*') if p.is_file()],
}
(ROOT / 'results/artifact_manifest.json').write_text(json.dumps(artifact_manifest, indent=2), encoding='utf-8')
output_file = str(ROOT / 'volatility_pipeline_artifacts.zip')
archive_base = Path(output_file[:-4])
shutil.make_archive(str(archive_base), 'zip', root_dir=ROOT, base_dir='results')
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)